# 🔥 Advanced · Habitat embodied navigation

> 🔥 **Advanced / heavy lab.** Put an agent in a photorealistic 3D house and learn PointGoal navigation.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **heavy — sim install + training** including downloads. Built on the official **[facebookresearch/habitat-lab](https://github.com/facebookresearch/habitat-lab)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Where agents (track AG) meet scene reconstruction & world models (track D): act inside a reconstructed 3D world.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | 1× T4 / 24 GB — small budget (10–50M steps) | many GPUs — DD-PPO scales to 8–64 |
| **Storage** | Habitat test scenes ~ small | HM3D ~ tens of GB, MP3D ~ 15 GB, Gibson; 50–150 GB disk |
| **Time** | 10–50M steps ~ hours–1 day on 1 GPU | PointNav SOTA = 2.5B steps, days on 64 GPUs |

**Full pipeline (scale-up):** multi-node DD-PPO (`torchrun` / SLURM), full scene dataset, billions of steps.

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi

## 1 · Install Habitat-Sim + Habitat-Lab (headless)

In [ ]:
!pip install -q habitat-sim --extra-index-url https://dl.fbaipublicfiles.com/habitat/habitat-sim/whl/cu121
!git clone --branch stable https://github.com/facebookresearch/habitat-lab.git
%cd habitat-lab
!pip install -q -e habitat-lab

## 2 · Download a test scene (Habitat test assets)

In [ ]:
!python -m habitat_sim.utils.datasets_download --uids habitat_test_scenes --data-path data/

## 3 · Train a PointGoal navigation agent (DD-PPO)

In [ ]:
!python -u habitat-baselines/habitat_baselines/run.py \
  --config-name=pointnav/ppo_pointnav_example.yaml \
  habitat_baselines.total_num_steps=20000

## Evaluate — Success rate & SPL
Navigation is scored by **Success rate** and **SPL** (Success weighted by Path Length). Run the baseline in eval mode — `... run.py --config-name=pointnav/ppo_pointnav_example.yaml habitat_baselines.evaluate=True` — to print Success/SPL over the validation episodes.

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/ropedia-<lab>")`.

## How this links to tracks A–D
Embodied agents live in reconstructed worlds:
- **D · Scene / world** act inside a 3D scene / map (SLAM, world models) · learned with RL (track AG).

## Troubleshooting & next steps
- Habitat is **install-heavy**; match the `habitat-sim` wheel to the runtime CUDA, run headless (EGL).
- Swap in **ObjectNav** / **ImageNav**; feed the agent a learned world model (DreamerV3 lab) or map (SLAM lab, track D).